# C3 — Conception évolutionnaire de molécules candidatesAnalyses et visualisations du projet. **Ce notebook est autonome** : ilinstalle ses dépendances, exécute les expériences et produit toutes lesfigures du rapport. Aucun fichier de résultats préalable n'est nécessaire.| Membre | Figures ||---|---|| [@guilhermelopo](https://github.com/guilhermelopo) | 1 (validité), 2 (médicaments connus), 5 (diversité) || [@lucaslapenda-cpu](https://github.com/lucaslapenda-cpu) | 3 (convergence), 4 (Pareto), 6 (meilleures molécules) |## Comment l'exécuter**Google Colab** — téléverser le dossier `groupe-molevo` puis exécuter« Exécuter tout ». La première cellule installe RDKit et selfies.**En local** — depuis `groupe-molevo/` :`pip install -r requirements.txt && jupyter lab notebooks/analise.ipynb`> Durée totale avec les paramètres par défaut : environ 3 à 6 minutes.> Les paramètres sont tous dans la cellule « Paramètres » ci-dessous :> un seul endroit à modifier pour refaire toutes les mesures.

In [ ]:
# Dependances. Sans effet si elles sont deja installees.import importlib, subprocess, sysfor paquet, module in [("rdkit", "rdkit"), ("selfies", "selfies"),                       ("matplotlib", "matplotlib"), ("pandas", "pandas")]:    if importlib.util.find_spec(module) is None:        print(f"installation de {paquet}...")        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet], check=True)import rdkit, selfiesprint(f"rdkit {rdkit.__version__} | selfies {selfies.__version__}")

In [ ]:
# Rend `src` importable, que le notebook soit lance depuis notebooks/,# depuis la racine du dossier de groupe, ou depuis Colab.import sysfrom pathlib import Pathfor candidat in [Path.cwd(), Path.cwd().parent, Path.cwd() / "groupe-molevo"]:    if (candidat / "src" / "representation.py").exists():        racine = candidat        breakelse:    raise FileNotFoundError(        "src/ introuvable. Lancez le notebook depuis groupe-molevo/ ou "        "notebooks/, ou televersez tout le dossier dans Colab."    )sys.path.insert(0, str(racine))print(f"racine du projet : {racine}")

In [ ]:
import matplotlib.pyplot as pltimport matplotlib as mpl# Style sobre et lisible en projection. Pas de couleur decorative :# chaque couleur porte une information.mpl.rcParams.update({    "figure.dpi": 110,    "figure.figsize": (7.5, 4.2),    "axes.spines.top": False,    "axes.spines.right": False,    "axes.grid": True,    "grid.alpha": 0.25,    "grid.linestyle": "-",    "font.size": 11,    "axes.titlesize": 12,    "axes.titleweight": "bold",    "legend.frameon": False,})BLEU, ROUGE, VERT, GRIS = "#2a56c6", "#b5362a", "#0c6f63", "#8893a3"

## ParamètresUn seul endroit à modifier. Les graines sont fixées : réexécuter le notebookdonne exactement les mêmes nombres, ce qui est la condition dereproductibilité exigée par le guide.

In [ ]:
GRAINES   = 5        # nombre de graines aleatoires (>= 5 pour le rapport)POP       = 60       # taille de la populationGENERATIONS = 60     # nombre de generationsPESO_QED  = 0.5      # poids du QED dans la somme ponderee (1.0 = plan B)N_MUTACOES = 2000    # mutations par representation, test de validiteGRAINE_VALIDITE = 12345BUDGET = POP * GENERATIONS   # evaluations par methode et par graineprint(f"budget : {BUDGET} evaluations par methode et par graine")print(f"total  : {BUDGET * 3 * GRAINES} evaluations sur l'ensemble du notebook")

## Figure 1 — Validité sous mutation : la justification du projetC'est la mesure qui justifie tout le reste. On applique la **mêmeperturbation conceptuelle** aux deux représentations — une altération d'uneunité (un caractère du SMILES, un jeton du SELFIES) — et on compte combiende fois le résultat est encore une molécule que RDKit sait lire.Si cet écart n'était pas large, il n'y aurait aucune raison d'utiliserSELFIES.

In [ ]:
import randomfrom rdkit import Chem, RDLoggerimport selfies as sfRDLogger.DisableLog("rdApp.*")MEDICAMENTS = {    "aspirine":      "CC(=O)Oc1ccccc1C(=O)O",    "paracetamol":   "CC(=O)Nc1ccc(O)cc1",    "ibuprofene":    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",    "cafeine":       "Cn1cnc2c1c(=O)n(C)c(=O)n2C",    "penicilline G": "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",    "atorvastatine": "CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CCC(O)CC(O)CC(=O)O",}rng = random.Random(GRAINE_VALIDITE)alphabet_selfies = sorted(sf.get_semantic_robust_alphabet())alphabet_smiles = list("CNOSFPcnos()[]=#123456+-")departs = list(MEDICAMENTS.values())valides_smiles = valides_selfies = 0casses = []for i in range(N_MUTACOES):    base = departs[i % len(departs)]    # SMILES : remplacer un caractere    p = rng.randrange(len(base))    mute = base[:p] + rng.choice(alphabet_smiles) + base[p + 1:]    if Chem.MolFromSmiles(mute) is not None:        valides_smiles += 1    elif len(casses) < 6:        casses.append((base, mute))    # SELFIES : remplacer un jeton    jetons = list(sf.split_selfies(sf.encoder(base)))    p = rng.randrange(len(jetons))    jetons[p] = rng.choice(alphabet_selfies)    if Chem.MolFromSmiles(sf.decoder("".join(jetons))) is not None:        valides_selfies += 1pct_smiles = 100 * valides_smiles / N_MUTACOESpct_selfies = 100 * valides_selfies / N_MUTACOESprint(f"{N_MUTACOES} mutations d'une unite par representation\n")print(f"  SMILES  : {pct_smiles:5.1f} %  ({valides_smiles} molecules valides)")print(f"  SELFIES : {pct_selfies:5.1f} %  ({valides_selfies} molecules valides)")print(f"\n  facteur : x{pct_selfies / max(pct_smiles, 0.05):.1f}")print("\nExemples de SMILES casses par une seule substitution :")for avant, apres in casses[:4]:    print(f"  {avant}\n  -> {apres}\n")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6))barres = ax.bar(["SMILES", "SELFIES"], [pct_smiles, pct_selfies],                color=[ROUGE, VERT], width=0.55)for barre, valeur in zip(barres, [pct_smiles, pct_selfies]):    ax.text(barre.get_x() + barre.get_width() / 2, valeur + 1.5,            f"{valeur:.1f} %", ha="center", fontweight="bold")ax.set_ylabel("mutations donnant une molecule valide")ax.set_ylim(0, 108)ax.set_title(f"Figure 1 — Validite sous mutation ({N_MUTACOES} essais par representation)")ax.set_axisbelow(True)plt.tight_layout()plt.savefig(racine / "notebooks" / "fig1_validite.png", dpi=150, bbox_inches="tight")plt.show()

**À interpréter dans le rapport.** La validité de SELFIES n'est pas uneperformance de l'algorithme : c'est une propriété de la grammaire, obtenuepar construction. Le chiffre de SMILES, lui, dépend de l'alphabet desubstitution choisi — précisez le vôtre, il est dans la cellule ci-dessus.

## Figure 2 — L'échelle réelle des deux notesAvant de lancer une recherche, il faut savoir ce que « bon » veut dire.On mesure QED et SA sur six médicaments réels, du plus simple(aspirine, paracétamol) au plus complexe (atorvastatine).C'est ce tableau qui permet de dire, plus tard, si une molécule générée a unenote plausible ou suspecte.

In [ ]:
import pandas as pdfrom src import objectives as objfrom src import representation as replignes = []for nom, smi in MEDICAMENTS.items():    n = obj.notas(rep.de_smiles(smi))    lignes.append({"molecule": nom, "QED": round(n.qed, 3),                   "SA": round(n.sa, 2), "sa_norm": round(n.sa_norm, 3),                   "smiles": smi})medicaments_df = pd.DataFrame(lignes).sort_values("QED", ascending=False)display(medicaments_df[["molecule", "QED", "SA", "sa_norm"]].reset_index(drop=True))print(f"\nQED   : de {medicaments_df.QED.min():.3f} a {medicaments_df.QED.max():.3f}")print(f"SA    : de {medicaments_df.SA.min():.2f} a {medicaments_df.SA.max():.2f}")print("\nRetenez cette fourchette : un QED genere au-dela de 0,85 est suspect,")print("pas excellent -- aucun de ces medicaments approuves n'y arrive.")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.4))ax.scatter(medicaments_df.QED, medicaments_df.SA, s=130, color=BLEU,           zorder=3, edgecolor="white", linewidth=1.5)for _, r in medicaments_df.iterrows():    ax.annotate(r.molecule, (r.QED, r.SA), textcoords="offset points",                xytext=(9, 4), fontsize=9.5, color="#333")ax.set_xlabel("QED  (plus haut = meilleur profil de medicament oral)")ax.set_ylabel("SA score  (plus BAS = plus facile a synthetiser)")ax.invert_yaxis()ax.set_title("Figure 2 — Six medicaments reels dans l'espace des deux objectifs")ax.set_axisbelow(True)plt.tight_layout()plt.savefig(racine / "notebooks" / "fig2_medicaments.png", dpi=150, bbox_inches="tight")plt.show()

L'axe SA est inversé pour que « en haut à droite » signifie « meilleur » surles deux axes, comme dans les figures de Pareto qui suivent.

## Figure 3 — Le résultat principal : GA contre les lignes de baseUne courbe d'aptitude qui monte ne prouve rien : n'importe quelle méthode quigarde son meilleur résultat produit une courbe qui monte. Ce qui prouvequelque chose, c'est la comparaison à budget égal.Trois méthodes, **le même nombre d'évaluations**, plusieurs graines :- **recherche aléatoire** — le plancher ;- **hill climbing** — monte vite puis se bloque sur un optimum local ;- **algorithme génétique** — population, croisement, élitisme.L'axe horizontal est en **évaluations**, jamais en générations : unegénération de GA coûte `POP` évaluations, une itération de hill climbing encoûte une.

In [ ]:
import statistics, timefrom src import baselines, gaexecutions = {"ga": [], "aleatoire": [], "hill": []}t_debut = time.time()for graine in range(GRAINES):    obj.limpar_cache()    t0 = time.time()    r_ga = ga.evoluir(semente=graine, tamanho_pop=POP,                      geracoes=GENERATIONS, peso_qed=PESO_QED)    r_al = baselines.busca_aleatoria(graine, obj.Orcamento(BUDGET), PESO_QED)    r_hc = baselines.hill_climbing(graine, obj.Orcamento(BUDGET), PESO_QED)    executions["ga"].append(r_ga)    executions["aleatoire"].append(r_al)    executions["hill"].append(r_hc)    print(f"graine {graine} : GA {r_ga.melhor_aptidao:.4f} | "          f"aleatoire {r_al.melhor_aptidao:.4f} | "          f"hill {r_hc.melhor_aptidao:.4f}   ({time.time() - t0:.0f} s)")print(f"\ntotal : {time.time() - t_debut:.0f} s")

In [ ]:
def agreger(resultats):    """Moyenne et ecart-type sur les graines, reechantillonnes sur une    grille commune d'evaluations (les methodes n'echantillonnent pas aux    memes points)."""    import numpy as np    grille = np.linspace(0, BUDGET, 60)    courbes = []    for r in resultats:        x = np.array(r.historico_avaliacoes, dtype=float)        y = np.array(r.historico_melhor, dtype=float)        courbes.append(np.interp(grille, x, y))    courbes = np.array(courbes)    return grille, courbes.mean(axis=0), courbes.std(axis=0)fig, ax = plt.subplots()styles = {"ga": (VERT, "-", "algorithme genetique"),          "hill": (ROUGE, "--", "hill climbing"),          "aleatoire": (GRIS, ":", "recherche aleatoire")}tableau = []for nom, (couleur, trait, etiquette) in styles.items():    x, moy, ecart = agreger(executions[nom])    ax.plot(x, moy, color=couleur, linestyle=trait, linewidth=2.2, label=etiquette)    ax.fill_between(x, moy - ecart, moy + ecart, color=couleur, alpha=0.15)    finaux = [r.melhor_aptidao for r in executions[nom]]    tableau.append({        "methode": etiquette,        "moyenne": round(statistics.fmean(finaux), 4),        "ecart-type": round(statistics.pstdev(finaux), 4),        "max": round(max(finaux), 4),        "distinctes": len({a.smiles for r in executions[nom]                           for a in r.arquivo.itens.values()}),    })ax.set_xlabel("evaluations de la fonction d'aptitude")ax.set_ylabel("meilleure aptitude atteinte")ax.set_title(f"Figure 3 — Convergence a budget egal ({GRAINES} graines, bande = ecart-type)")ax.legend(loc="lower right")ax.set_axisbelow(True)plt.tight_layout()plt.savefig(racine / "notebooks" / "fig3_convergence.png", dpi=150, bbox_inches="tight")plt.show()resume_df = pd.DataFrame(tableau)display(resume_df)

**La question à se poser honnêtement.** L'écart entre le GA et la recherchealéatoire dépasse-t-il l'écart-type ? Si non, le GA n'évolue rien, et lacause est presque toujours un bug : aptitude calculée mais non utilisée dansla sélection, taux de mutation si élevé qu'il détruit tout héritage, oucroisement qui renvoie des chaînes vides.Un résultat négatif mesuré avec soin vaut mieux qu'un résultat positif malmesuré. Le critère d'évaluation du cours valorise explicitement « l'analysedes échecs et des limites ».

## Figure 4 — Le front de ParetoLes deux objectifs se contredisent : le QED favorise des molécules de taillemoyenne avec des cycles aromatiques, et ces systèmes sont précisément ce quele SA pénalise. Il n'existe donc pas *une* meilleure molécule, mais un**ensemble de compromis**.**A domine B** quand A est au moins aussi bon sur *tous* les objectifs etstrictement meilleur sur *au moins un*. Le front de Pareto est l'ensemble desnon dominés.> Attention à l'intuition : un point médiocre sur les deux critères peut> rester non dominé. Il ne sort du front que s'il perd sur **tous** les> objectifs à la fois. C'est pourquoi les fronts sont souvent plus grands> qu'on ne l'attend.

In [ ]:
from src import paretotous = [a for r in executions["ga"] for a in r.arquivo.itens.values()]points = [(a.qed, a.sa_norm) for a in tous]indices = pareto.indices_nao_dominados(points)front = [tous[i] for i in indices]hv = pareto.hipervolume_2d([(a.qed, a.sa_norm) for a in front])print(f"molecules distinctes evaluees par le GA : {len({a.smiles for a in tous})}")print(f"front de Pareto : {len(front)} molecules")print(f"hypervolume (reference 0,0) : {hv:.4f}")front_df = pd.DataFrame([    {"QED": round(a.qed, 3), "SA": round(a.sa, 2), "smiles": a.smiles}    for a in sorted(front, key=lambda a: a.qed, reverse=True)])display(front_df.head(10))

In [ ]:
import numpy as npfig, ax = plt.subplots(figsize=(7.2, 5))qed_tous = np.array([a.qed for a in tous])san_tous = np.array([a.sa_norm for a in tous])masque = np.zeros(len(tous), dtype=bool)masque[indices] = Trueax.scatter(qed_tous[~masque], san_tous[~masque], s=12, color=GRIS,           alpha=0.35, label=f"dominees ({(~masque).sum()})")ordre = np.argsort(qed_tous[masque])xf, yf = qed_tous[masque][ordre], san_tous[masque][ordre]ax.plot(xf, yf, color=BLEU, linewidth=1.4, linestyle="--", alpha=0.7, zorder=2)ax.scatter(xf, yf, s=55, color=BLEU, zorder=3, edgecolor="white",           linewidth=1.2, label=f"front de Pareto ({len(front)})")# Les medicaments reels, pour situer l'echelle.ax.scatter(medicaments_df.QED, medicaments_df.sa_norm, marker="*", s=240,           color=ROUGE, zorder=4, edgecolor="white", linewidth=0.8,           label="medicaments reels")ax.set_xlabel("QED  →")ax.set_ylabel("facilite de synthese (sa_norm)  →")ax.set_title(f"Figure 4 — Front de Pareto, hypervolume = {hv:.3f}")ax.legend(loc="lower left")ax.set_axisbelow(True)plt.tight_layout()plt.savefig(racine / "notebooks" / "fig4_pareto.png", dpi=150, bbox_inches="tight")plt.show()

Les étoiles rouges sont les six médicaments réels. **Si le front généré passelargement au-dessus d'eux, c'est un signal d'alerte, pas une victoire :**cela signifie que l'optimiseur a trouvé un recoin des deux formules où ellesdonnent de bonnes notes à des structures qu'aucun chimiste ne reconnaîtrait.La figure 6 montre ces molécules.

## Figure 5 — L'effondrement de la diversitéL'algorithme génétique converge, et c'est en partie le problème. Aprèsquelques dizaines de générations, la population est composée de descendantsdu même gagnant : la courbe d'aptitude est excellente et le résultat n'estplus une recherche.Pour la découverte de médicaments, c'est un échec déguisé. L'objectif n'ajamais été *une* molécule, mais un ensemble de candidats différents, parceque la plupart échoueront plus tard pour des raisons qu'aucune de cesformules ne prévoit.Deux mesures complémentaires :- **molécules inédites par génération** — l'exploration réelle ;- **diversité interne** (1 − Tanimoto moyen) de la population finale.

In [ ]:
from collections import Counterfrom src import diversity# Molecules inedites par generation, sur la premiere graine.r0 = executions["ga"][0]inedites = Counter(a.geracao for a in r0.arquivo.itens.values())gens = sorted(inedites)valeurs = [inedites[g] for g in gens]# Diversite interne : population initiale vs finale.import random as _randompop_init = rep.populacao_inicial(_random.Random(0), POP)smi_init = [s for s in (rep.canonico(c) for c in pop_init) if s]smi_fin = [s for s in (rep.canonico(c) for c in r0.populacao_final) if s]div_init = diversity.diversidade_interna(sorted(set(smi_init)))div_fin = diversity.diversidade_interna(sorted(set(smi_fin)))sq_init = diversity.contar_esqueletos(sorted(set(smi_init)))sq_fin = diversity.contar_esqueletos(sorted(set(smi_fin)))print(f"population initiale : {len(set(smi_init))} distinctes, "      f"diversite {div_init:.3f}, {sq_init} squelettes")print(f"population finale   : {len(set(smi_fin))} distinctes, "      f"diversite {div_fin:.3f}, {sq_fin} squelettes")print(f"\narchive totale : {len(r0.arquivo)} molecules distinctes conservees")print("C'est l'archive qui sauve le resultat : meme si la population finit en")print("clones, tout ce qui a ete trouve en chemin reste disponible.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4),                               gridspec_kw={"width_ratios": [2, 1]})ax1.bar(gens, valeurs, color=BLEU, width=0.9)ax1.set_xlabel("generation")ax1.set_ylabel("molecules inedites trouvees")ax1.set_title("Exploration par generation")ax1.set_axisbelow(True)x = ["initiale", "finale"]ax2.bar(x, [div_init, div_fin], color=[GRIS, ROUGE], width=0.55)for i, v in enumerate([div_init, div_fin]):    ax2.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")ax2.set_ylabel("diversite interne (1 - Tanimoto moyen)")ax2.set_ylim(0, 1.08)ax2.set_title("Population")ax2.set_axisbelow(True)fig.suptitle("Figure 5 — L'exploration s'eteint a mesure que l'aptitude monte",             fontweight="bold", y=1.02)plt.tight_layout()plt.savefig(racine / "notebooks" / "fig5_diversite.png", dpi=150, bbox_inches="tight")plt.show()

## Figure 6 — Les molécules, et la loi de Goodhart« Quand une mesure devient un objectif, elle cesse d'être une bonne mesure. »On affiche deux grilles :1. les meilleures selon l'**aptitude combinée** — ce que l'algorithme   cherchait vraiment ;2. les meilleures selon le **QED seul** — et c'est là que le problème se voit.Ces structures sont la pièce à conviction de la section « limites » durapport. Les montrer à la soutenance démontre plus de maîtrise que n'importequel bon chiffre.

In [ ]:
from rdkit.Chem import Drawdef grille(achados, titre, n=6):    choisis = achados[:n]    mols = [Chem.MolFromSmiles(a.smiles) for a in choisis]    legendes = [f"QED {a.qed:.3f} | SA {a.sa:.2f}" for a in choisis]    print(f"\n{titre}")    for a in choisis:        print(f"  QED {a.qed:.3f}  SA {a.sa:.2f}  {a.smiles}")    return Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 230),                                legends=legendes)meilleures_aptitude = sorted(tous, key=lambda a: a.aptidao, reverse=True)grille(meilleures_aptitude, "Meilleures selon l'aptitude combinee (QED + synthese)")

In [ ]:
meilleures_qed = sorted(tous, key=lambda a: a.qed, reverse=True)grille(meilleures_qed, "Meilleures selon le QED SEUL -- a examiner d'un oeil critique")

In [ ]:
# Comparaison chiffree : optimiser le QED seul coute-t-il en synthetisabilite ?top_q = meilleures_qed[:10]top_a = meilleures_aptitude[:10]print(f"{'critere':<34}{'QED moyen':>11}{'SA moyen':>10}")print(f"{'top 10 par QED seul':<34}"      f"{statistics.fmean(a.qed for a in top_q):>11.3f}"      f"{statistics.fmean(a.sa for a in top_q):>10.2f}")print(f"{'top 10 par aptitude combinee':<34}"      f"{statistics.fmean(a.qed for a in top_a):>11.3f}"      f"{statistics.fmean(a.sa for a in top_a):>10.2f}")print(f"{'six medicaments reels':<34}"      f"{medicaments_df.QED.mean():>11.3f}{medicaments_df.SA.mean():>10.2f}")

## Vérificateur indépendantLivrable obligatoire du cours. Il relit tout ce que la recherche a produit etrevérifie par un chemin de code propre, sans réutiliser le cache ni lesfonctions employées pendant l'exécution.L'erreur qu'il existe pour attraper est banale et dévastatrice : rapporter unemolécule avec une note qui n'est pas la sienne — index échangé lors d'un tri,cache périmé, enregistrement écrit avant la dernière mutation. Le graphiquereste beau et le résultat est faux.

In [ ]:
from src import verifierregistre = [a.como_dict() for a in tous]problemes = verifier.verificar(registre)problemes_front = verifier.verificar_frente([    {"smiles": a.smiles, "qed": a.qed, "sa_norm": a.sa_norm} for a in front])print(f"molecules verifiees : {len(registre)}")print(f"problemes dans le registre : {len(problemes)}")print(f"problemes dans le front    : {len(problemes_front)}")if not problemes and not problemes_front:    print("\nOK -- 0 probleme. Chaque note rapportee a ete recalculee et confirmee.")else:    for p in (problemes + problemes_front)[:15]:        print(f"  {p}")

In [ ]:
# Preuve que le verificateur fonctionne vraiment : on falsifie une note# et on verifie qu'il la detecte. Un verificateur qui ne trouve jamais rien# n'est pas une preuve de qualite -- il peut simplement etre casse.import copysabote = copy.deepcopy(registre[:50])sabote[0]["qed"] = 0.99detectes = verifier.verificar(sabote)print(f"note falsifiee sur 1 molecule sur 50 -> {len(detectes)} probleme(s) detecte(s)")for p in detectes:    print(f"  {p}")assert any(p.tipo == "qed_divergente" for p in detectes), \    "le verificateur n'a PAS detecte la falsification"print("\nOK -- le verificateur detecte bien une note falsifiee.")

## Sauvegarde des résultatsLes mesures sont écrites sur disque pour que le README et les slides citentexactement les mêmes nombres que ce notebook.

In [ ]:
import jsonmesures = {    "parametres": {        "graines": GRAINES, "pop": POP, "generations": GENERATIONS,        "peso_qed": PESO_QED, "budget_par_methode": BUDGET,        "n_mutations_validite": N_MUTACOES,    },    "validite": {"smiles_pct": round(pct_smiles, 2),                 "selfies_pct": round(pct_selfies, 2)},    "medicaments": medicaments_df.drop(columns=["smiles"]).to_dict("records"),    "comparaison": tableau,    "pareto": {"taille_front": len(front), "hypervolume": round(hv, 4),               "distinctes_evaluees": len({a.smiles for a in tous})},    "diversite": {"initiale": round(div_init, 4), "finale": round(div_fin, 4),                  "squelettes_initiaux": sq_init, "squelettes_finaux": sq_fin,                  "taille_archive": len(r0.arquivo)},    "verification": {"molecules": len(registre), "problemes": len(problemes)},}chemin = racine / "notebooks" / "mesures.json"with open(chemin, "w", encoding="utf-8") as f:    json.dump(mesures, f, ensure_ascii=False, indent=2)print(f"ecrit : {chemin}")print(json.dumps(mesures, ensure_ascii=False, indent=2)[:1200])

## Conclusions**À COMPLÉTER par le groupe** — une fois le notebook exécuté, remplir avecles nombres réels obtenus ci-dessus :1. **La représentation.** La validité passe de ___ % (SMILES) à ___ %   (SELFIES) sous la même perturbation. C'est ce qui rend le croisement et la   mutation utilisables sans rejet.2. **La recherche.** À budget égal de ___ évaluations et sur ___ graines, le   GA atteint ___ ± ___ contre ___ ± ___ pour la recherche aléatoire et   ___ ± ___ pour le hill climbing. Conclusion : ___3. **Le multiobjectif.** Le front compte ___ molécules, hypervolume ___.   Interprétation : ___4. **La diversité.** Elle passe de ___ à ___ entre la population initiale et   la finale ; l'archive conserve ___ molécules distinctes. Ce que cela   implique pour l'usage réel : ___5. **Les limites.** Les molécules au QED le plus élevé (figure 6) montrent   que ___. Ce que nous ne prétendons pas avoir fait : ___### Ce que nous ne prétendons pas avoir faitCe travail propose des structures qui obtiennent de bonnes notes sur deuxformules statistiques. Il ne découvre pas de médicaments. QED et SA sont desfiltres grossiers : le premier mesure une ressemblance de profil physique avecles médicaments oraux connus, le second la fréquence de fragments dans desbases de composés, sans simuler aucune réaction. Une campagne réelles'appuierait sur la simulation d'amarrage sur une cible, des modèles d'ADME etde toxicité, et le jugement de chimistes médicinaux.